# AI Safety Advisor: Policy Risk Extraction

This notebook demonstrates the Asago Policy Mapper, which extracts AI-related risks from policy documents using the [IBM AI Atlas Nexus](https://github.com/IBM/ai-atlas-nexus) risk taxonomy.

The pipeline uses hybrid retrieval (BM25 + semantic embeddings + cross-encoder reranking) to match document sections against ~480 risks across multiple taxonomies, then uses an LLM to judge borderline candidates and ground accepted risks with evidence quotes from the document.

**What you'll see:**
1. Extract risks from a policy document
2. View matched risks with confidence scores and taxonomy breakdown
3. Inspect evidence grounding — which document sections support each risk
4. Review causal chains — threat, vulnerability, consequence, and impact
5. Explore recommended mitigations for each risk
6. Identify document sections with no risk coverage

---

### Prerequisites

Before running this notebook, you need:

1. **Python virtual environment** — install dependencies:
   ```bash
      uv sync --extra policy-mapper
   ```
   In Jupyter, select this venv as your kernel: **Kernel → Change Kernel** → choose the `.venv` Python interpreter.

2. **`ai-atlas-nexus`** — a local clone of the [IBM AI Atlas Nexus](https://github.com/IBM/ai-atlas-nexus) risk taxonomy, checked out at the tag the installed library expects:
   ```bash
   git clone https://github.com/IBM/ai-atlas-nexus && git -C ai-atlas-nexus checkout v1.2.3
   ```
   The clone must match the pinned `ai-atlas-nexus` version (see the root `pyproject.toml` override). A newer checkout fails schema validation, since its data files contain fields the installed models don't know.

3. **A vLLM endpoint** serving `gemma-4-26b-a4b-it` (MoE, 26B total / 4B active, fits on a single L40S). This is the only model tested and supported for this pipeline.

4. **Policy documents** — examples are bundled in `asago-policy-mapper/policy_examples/`, or upload your own (PDF, DOCX, HTML, Markdown)

Set these environment variables before launching Jupyter, or edit the config cell directly:
```bash
export POLICY_MAPPER_BASE_URL="https://vllm-route-user-aaracan.apps.ocp.cloud.rhai-tmm.dev/v1"
export POLICY_MAPPER_MODEL="gemma-4-26b-a4b-it"      # must match the model name in vLLM
export NEXUS_BASE_DIR="/Users/aaracan/Visual Studio Code Projects/asago-examples/ai-atlas-nexus"
export POLICY_EXAMPLES_DIR="/Users/aaracan/Visual Studio Code Projects/asago-examples/asago-policy-mapper/policy_examples"
```

## 1. Configuration

Edit the values below if environment variables are not set.

> **Note:** Extraction typically takes **5-10 minutes** depending on document length and model endpoint capacity. The pipeline performs multiple LLM passes for judging, grounding, and causal chain synthesis to ensure accurate results.

In [ ]:
import os

BASE_URL = os.environ.get("POLICY_MAPPER_BASE_URL", "https://vllm-route-user-aaracan.apps.ocp.cloud.rhai-tmm.dev/v1")
MODEL = os.environ.get("POLICY_MAPPER_MODEL", "gemma-4-26b-a4b-it")
API_KEY = os.environ.get("POLICY_MAPPER_API_KEY", "none")
NEXUS_BASE_DIR = os.environ.get("NEXUS_BASE_DIR", "/Users/aaracan/Visual Studio Code Projects/asago-examples/ai-atlas-nexus")
POLICY_EXAMPLES_DIR = os.environ.get("POLICY_EXAMPLES_DIR", "/Users/aaracan/Visual Studio Code Projects/asago-examples/asago-policy-mapper/policy_examples")

print(f"LLM endpoint:     {BASE_URL}")
print(f"Model:            {MODEL}")
print(f"Nexus dir:        {NEXUS_BASE_DIR}")
print(f"Policy examples:  {POLICY_EXAMPLES_DIR}")

LLM endpoint:     http://vllm-gemma-4-26b-a4b-it.user-aaracan.svc.cluster.local:8000/v1
Model:            gemma-4-26b-a4b-it
Nexus dir:        /Users/aaracan/Visual Studio Code Projects/ai-atlas-nexus
Policy examples:  /Users/aaracan/Visual Studio Code Projects/asago-examples/asago-policy-mapper/policy_examples


## 2. Select Policy Document

Choose a bundled policy example from the dropdown, or upload your own document in the cell below.

In [11]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

examples_dir = Path(POLICY_EXAMPLES_DIR)
example_files = sorted(f.name for f in examples_dir.glob("*") if f.suffix in (".pdf", ".docx", ".md", ".html"))
print(f"Found {len(example_files)} policy examples\n")

policy_dropdown = widgets.Dropdown(
    options=example_files,
    description="Policy:",
    style={"description_width": "60px"},
    layout=widgets.Layout(width="500px"),
)
display(policy_dropdown)

# Mutable container so upload cell can override
_custom_policy_path = [None]

def get_policy_path():
    """Return custom upload path if set, otherwise the dropdown selection."""
    if _custom_policy_path[0] is not None:
        return _custom_policy_path[0]
    return examples_dir / policy_dropdown.value

print("To upload your own document, run the next cell.")

Found 27 policy examples



Dropdown(description='Policy:', layout=Layout(width='500px'), options=('amadeus.pdf', 'ars.md', 'aus-gov.pdf',…

To upload your own document, run the next cell.


In [4]:
# Optional: upload a custom policy document
# 1. Click "Upload" and select your file
# 2. Click "Use uploaded file" to save it and override the dropdown selection

import tempfile

upload = widgets.FileUpload(accept=".pdf,.docx,.html,.md", multiple=False)
save_btn = widgets.Button(description="Use uploaded file", button_style="primary", icon="check")
status = widgets.Output()

_upload_dir = Path(tempfile.mkdtemp(prefix="policy_upload_"))

def on_save_click(b):
    status.clear_output()
    with status:
        if not upload.value:
            print("No file selected — please click Upload first.")
            return
        info = upload.value[0]
        name = info["name"]
        content = info["content"]
        save_path = _upload_dir / name
        save_path.write_bytes(content.tobytes() if hasattr(content, "tobytes") else content)
        _custom_policy_path[0] = save_path
        print(f"Uploaded and selected: {name}")
        print("This will be used instead of the dropdown selection.")

save_btn.on_click(on_save_click)
display(widgets.HBox([upload, save_btn]), status)

Output()

## 3. Setup

In [12]:
from ai_atlas_nexus import AIAtlasNexus

from asago_policy_mapper.extract.mitigations import (
    build_action_descriptions,
    build_risk_crossmap,
    enrich_with_mitigations,
    load_mitigation_index,
    load_risk_consequences,
    load_risk_threats,
)
from asago_policy_mapper.extract.models import RetrievalConfig
from asago_policy_mapper.extract.pipeline import run_extraction
from asago_policy_mapper.llm import LLMConfig, TokenTracker, create_client

from ai_atlas_nexus import AIAtlasNexus

from asago_policy_mapper.extract.mitigations import (
    build_action_descriptions,
    build_risk_crossmap,
    enrich_with_mitigations,
    load_mitigation_index,
    load_risk_consequences,
    load_risk_threats,
)
from asago_policy_mapper.extract.models import RetrievalConfig
from asago_policy_mapper.extract.pipeline import run_extraction
from asago_policy_mapper.llm import LLMConfig, TokenTracker, create_client

In [13]:
EXCLUDED_TAXONOMIES = {
    "mit-ai-risk-repository-causal",
    "ibm-granite-guardian",
    "nist-ai-rmf",
    "owasp-llm-2.0",
    "ailuminate-v1.0",
    "owasp-asi",
    "shieldgemma-taxonomy",
}

nexus = AIAtlasNexus(base_dir=NEXUS_BASE_DIR)
all_risks = [
    r for r in nexus.get_all_risks()
    if getattr(r, "isDefinedByTaxonomy", "") not in EXCLUDED_TAXONOMIES
]
print(f"Loaded {len(all_risks)} risks from AI Atlas Nexus")

[2026-10-08 11:15:07:734] - INFO - AIAtlasNexus - YAML ignored: /Users/aaracan/Visual Studio Code Projects/ai-atlas-nexus/docs/examples/notebooks/example_samples/sample_additional_schema.yaml. Failed to load. string indices must be integers, not 'str'
[2026-10-08 11:15:08:935] - INFO - AIAtlasNexus - YAML ignored: /Users/aaracan/Visual Studio Code Projects/ai-atlas-nexus/src/ai_atlas_nexus/ai_risk_ontology/schema/ai_intrinsic.yaml. Failed to load. string indices must be integers, not 'str'
[2026-10-08 11:15:08:937] - INFO - AIAtlasNexus - YAML ignored: /Users/aaracan/Visual Studio Code Projects/ai-atlas-nexus/src/ai_atlas_nexus/ai_risk_ontology/schema/ai_risk.yaml. Failed to load. string indices must be integers, not 'str'
[2026-10-08 11:15:08:938] - INFO - AIAtlasNexus - YAML ignored: /Users/aaracan/Visual Studio Code Projects/ai-atlas-nexus/src/ai_atlas_nexus/ai_risk_ontology/schema/ai_aiuc.yaml. Failed to load. string indices must be integers, not 'str'
[2026-10-08 11:15:08:939] - I

ValidationError: 547 validation errors for Container
datasets.18.isProvidedBy
  Extra inputs are not permitted [type=extra_forbidden, input_value='cais', input_type=extended_str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
datasets.23.isProvidedBy
  Extra inputs are not permitted [type=extra_forbidden, input_value='cais', input_type=extended_str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
datasets.24.isProvidedBy
  Extra inputs are not permitted [type=extra_forbidden, input_value='TIGER-lab', input_type=extended_str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
datasets.41.isProvidedBy
  Extra inputs are not permitted [type=extra_forbidden, input_value='codeparrot', input_type=extended_str]
    For further information visit https://errors.pydantic.dev/2.13/v/extra_forbidden
datasets.42.isProvidedBy
  Extra inputs are not permitted [type=extra_forbidden, input_value

## 5. Run the Extraction Pipeline

This runs the full pipeline:
1. **Parse** the policy document (PDF/DOCX/HTML/Markdown)
2. **Chunk** into ~512-token segments
3. **Retrieve** candidate risks per chunk (BM25 + semantic + cross-encoder)
4. **Judge** borderline candidates via LLM
5. **Ground** accepted risks with evidence quotes
6. **Expand** to sibling risks and cross-taxonomy mappings
7. **Synthesize** causal chains (threat, vulnerability, consequence, impact)

In [14]:
config = LLMConfig(base_url=BASE_URL, model=MODEL, api_key=API_KEY)
tracker = TokenTracker()
client = create_client(config, tracker=tracker)
retrieval = RetrievalConfig()

policy_path = get_policy_path()
print(f"Extracting risks from: {policy_path.name}")

result = run_extraction(
    documents=[policy_path],
    client=client,
    config=config,
    risks=all_risks,
    retrieval=retrieval,
)

result.token_usage = tracker.to_dict()

# Enrich with mitigations
mitigation_index = load_mitigation_index()
if mitigation_index:
    action_descs = build_action_descriptions(NEXUS_BASE_DIR)
    risk_crossmap = build_risk_crossmap(NEXUS_BASE_DIR)
    risk_threats = load_risk_threats()
    risk_consequences = load_risk_consequences()
    enrich_with_mitigations(
        result.risks, mitigation_index, action_descs,
        risk_crossmap, risk_threats, risk_consequences,
    )

print(f"\nExtraction complete:")
print(f"  {len(result.risks)} risks identified")
print(f"  {result.retrieval_stats.total_chunks} document chunks processed")
print(f"  {tracker.total_tokens:,} tokens used ({tracker.calls} LLM calls)")

Extracting risks from: dhs-gov.md


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Query generation failed for group section=U.S. Department of Homeland Security — Artificial Intelligence Use Directive chunks=[0], falling back
Query generation failed for group section=Principles chunks=[1], falling back
Query generation failed for group section=FEMA — Federal Emergency Management Agency chunks=[7], falling back
Query generation failed for group section=Governance chunks=[11], falling back
Query generation failed for group section=CBP — Customs and Border Protection chunks=[4], falling back
Query generation failed for group section=Training chunks=[13], falling back
Query generation failed for group section=USCIS — U.S. Cit

InstructorRetryException: <failed_attempts>

<generation number="1">
<exception>
    Connection error.
</exception>
<completion>
    None
</completion>
</generation>

</failed_attempts>

<last_exception>
    Connection error.
</last_exception>

## 6. Extracted Risks Overview

Summary of all risks identified in the policy document, grouped by taxonomy.

In [ ]:
import pandas as pd
from collections import Counter

pd.set_option("display.max_rows", None)

risks_data = []
for r in result.risks:
    risks_data.append({
        "Risk ID": r.risk_id,
        "Name": r.risk_name,
        "Taxonomy": r.taxonomy or r.risk_id.split("-")[0],
        "Confidence": f"{r.confidence:.2f}",
        "Grounding": r.grounding_confidence or "N/A",
        "Evidence": len(r.evidence),
        "Mitigations": len(r.mitigations),
    })

df = pd.DataFrame(risks_data)

# Taxonomy breakdown
print("Risks by taxonomy:")
for taxonomy, count in Counter(d["Taxonomy"] for d in risks_data).most_common():
    print(f"  {taxonomy}: {count}")
print()

df

## 7. Risk Details: Evidence & Causal Chains

For each identified risk, inspect:
- **Evidence** — the exact document passages that support the risk identification
- **Causal chain** — how the risk manifests: threat source, vulnerability, consequence, and impact
- **Mitigations** — recommended actions to address the risk

In [ ]:
from IPython.display import display, HTML

def render_risk_detail(risk, index):
    html = f'<div style="border:1px solid #ddd; padding:16px; margin:8px 0; border-radius:8px; background:#fafafa;">'
    html += f'<h3 style="margin-top:0;">#{index + 1} {risk.risk_name}</h3>'
    html += f'<p><b>ID:</b> <code>{risk.risk_id}</code> &nbsp; '
    html += f'<b>Confidence:</b> {risk.confidence:.2f} &nbsp; '
    html += f'<b>Grounding:</b> {risk.grounding_confidence or "N/A"}</p>'
    html += f'<p style="color:#555;">{risk.risk_description}</p>'

    # Evidence
    if risk.evidence:
        html += '<h4>Evidence</h4>'
        for ev in risk.evidence:
            loc = f"Page {ev.page}" if ev.page else ""
            if ev.section:
                loc += f" \u2014 {ev.section}" if loc else ev.section
            html += f'<div style="background:#e8f4f8; padding:8px; margin:4px 0; border-left:3px solid #2196F3; font-size:0.9em;">'
            if loc:
                html += f'<div style="color:#666; font-size:0.85em;">{loc}</div>'
            html += f'<em>"{ev.text}"</em></div>'

    # Causal chain
    causal_fields = [
        ("Threat", risk.threat),
        ("Threat Source", risk.threat_source),
        ("Vulnerability", risk.vulnerability),
        ("Consequence", risk.consequence),
        ("Impact", risk.impact),
    ]
    if any(v for _, v in causal_fields):
        html += '<h4>Causal Chain</h4><table style="width:100%; font-size:0.9em;">'
        for label, value in causal_fields:
            if value:
                html += f'<tr><td style="padding:4px 8px; font-weight:bold; width:120px; vertical-align:top;">{label}</td>'
                html += f'<td style="padding:4px 8px;">{value}</td></tr>'
        html += '</table>'

    # Mitigations
    if risk.mitigations:
        html += f'<h4>Mitigations ({len(risk.mitigations)})</h4>'
        html += '<ul style="font-size:0.9em;">'
        for m in risk.mitigations[:5]:
            desc = f" \u2014 {m.description}" if m.description else ""
            source = f' <span style="color:#888;">({m.source})</span>'
            html += f'<li><b>{m.action_name or m.action_id}</b>{desc}{source}</li>'
        if len(risk.mitigations) > 5:
            html += f'<li style="color:#888;">... and {len(risk.mitigations) - 5} more</li>'
        html += '</ul>'

    html += '</div>'
    return html

# Render all risks
for i, risk in enumerate(result.risks):
    display(HTML(render_risk_detail(risk, i)))

## 8. Unmapped Document Sections

Document chunks where the pipeline did not identify any risks. These sections may contain content that is not risk-related, or may indicate gaps in taxonomy coverage.

In [ ]:
unmapped = [c for c in result.chunks if not c.accepted_risk_ids]

print(f"{len(unmapped)} of {len(result.chunks)} chunks have no mapped risks\n")

if unmapped:
    unmapped_html = '<div style="border:1px solid #ddd; padding:16px; border-radius:8px;">'
    unmapped_html += f'<h4 style="margin-top:0;">Unmapped Chunks ({len(unmapped)} of {len(result.chunks)})</h4>'
    for chunk in unmapped:
        loc = f"Page {chunk.page}" if chunk.page else ""
        if chunk.section:
            loc += f" \u2014 {chunk.section}" if loc else chunk.section
        unmapped_html += f'<div style="background:#fff3e0; padding:8px; margin:4px 0; border-left:3px solid #FF9800; font-size:0.9em;">'
        if loc:
            unmapped_html += f'<div style="color:#666; font-size:0.85em;">{loc}</div>'
        unmapped_html += f'{chunk.text_preview}</div>'
    unmapped_html += '</div>'
    display(HTML(unmapped_html))
else:
    print("All document chunks have at least one mapped risk.")

## 9. Pipeline Statistics

In [ ]:
stats = result.retrieval_stats
timing = stats.timing_ms

print("Retrieval Statistics")
print(f"  Document chunks:      {stats.total_chunks}")
print(f"  Candidates retrieved: {stats.total_candidates_retrieved}")
print(f"  Auto-accepted:        {stats.auto_accepted}")
print(f"  LLM-judged:           {stats.llm_judged}")
print(f"  Grounding-filtered:   {stats.grounding_filtered}")
print(f"  Final risks:          {len(result.risks)}")
print()
print("Token Usage")
print(f"  Prompt tokens:     {tracker.prompt_tokens:,}")
print(f"  Completion tokens: {tracker.completion_tokens:,}")
print(f"  Total tokens:      {tracker.total_tokens:,}")
print(f"  LLM calls:         {tracker.calls}")
print()
print("Timing")
for stage, ms in timing.items():
    if ms > 0:
        print(f"  {stage.replace('_ms', ''):20s} {ms / 1000:.1f}s")

## 10. Export Results

Save the extraction results and generate the annotation view.

In [ ]:
import json

output_dir = Path("output")
output_dir.mkdir(exist_ok=True)

result_data = result.model_dump()

json_path = output_dir / "risk-extraction.json"
json_path.write_text(json.dumps(result_data, indent=2))
print(f"Results saved to {json_path}")

# Generate annotation view
from asago_policy_mapper.extract.annotate import prepare_annotation_data
from asago_policy_mapper.extract.report import build_annotation_report

annotation_data = prepare_annotation_data(result_data, policy_path)
ann_path = build_annotation_report(annotation_data, output_dir / "risk-annotation.html")

summary = annotation_data["summary"]
print(f"Annotation view: {summary['total_risks']} risks, {summary['matched_spans']}/{summary['total_evidence_spans']} evidence spans matched")
print(f"Saved to {ann_path}")
print(f"\nOpen {ann_path} in a browser to view risks highlighted on the document.")